## 批量处理：循环所有指定的 sif

与上面各 cell **完全同一套参数**，只是不画图、不打印中间过程；数据、图和 `summary.txt` 照常落盘到
`<sif 文件名>_fret/`。

每个文件依次做：读 sif → 切通道 → 双通道定位（带缓存）→ 估漂移 + 校正 → 连 emitter →
取双通道 trace → 扫台阶 → 标定 γ/α/β → 算 E → 存盘。

In [ ]:
# ============ 批量处理: 循环所有指定的 sif ============
# 每个文件逐阶段打进度(带耗时戳 + 缓存命中情况), 流程内部的 print 全部丢掉。
# 每个视野只落盘**给合作者的成品**(不再写 fretcal.npz / gamma.png / summary.txt):
#   <sif 同目录>/<文件名>_fret{OUT_TAG}/
#       fret.csv                        逐分子 E 与 γ/α/β
#       E_distribution.png              E 分布 + 两个交叉检查
#       good_traces_all.png             通过 7 条判据的 trace 总览
#       traces_all.csv                  全部(未筛选)双通道 trace, 一行 = 一分子一帧
#       good_traces.csv                 通过 7 条判据的 trace 全帧 (同上格式)
import os
import gc
import io
import sys
import time
import contextlib
import csv

import numpy as np
import matplotlib.pyplot as plt

import smlm.multiplex as mp
import smlm.fret as fr
import smlm.trace as tr
from smlm.camera import EMCCD
from smlm.imgprocess import Pipeline, Crop, SelectFrames
from smlm.io import load_raw
from smlm.emitter import analyze_emitters_sm

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ---------------------------------------------------------------- 要处理的文件
SIF_FILES = [
    r"G:\PacBio\近观DNA_FRET\20261109\sample1_532nm_2mW_0p1S_2000frames_fov5.sif",
    r"G:\PacBio\近观DNA_FRET\20261109\sample1_532nm_2mW_0p1S_2000frames_fov6.sif",
    r"G:\PacBio\近观DNA_FRET\20261109\sample2_532nm_2mW_0p1S_2000frames_fov1.sif",
    r"G:\PacBio\近观DNA_FRET\20261109\sample2_532nm_2mW_0p1S_2000frames_fov2.sif",
    r"G:\PacBio\近观DNA_FRET\20261109\sample2_532nm_2mW_0p1S_2000frames_fov3.sif",
    r"G:\PacBio\近观DNA_FRET\20261109\sample4_532nm_2mW_0p1S_2000frames_fov2.sif",
    r"G:\PacBio\近观DNA_FRET\20261109\sample4_532nm_2mW_0p1S_2000frames_fov3.sif",
    r"G:\PacBio\近观DNA_FRET\20261109\sample5_532nm_2mW_0p1S_2000frames_fov1.sif",
    r"G:\PacBio\近观DNA_FRET\20261109\sample5_532nm_2mW_0p1S_2000frames_fov2.sif",
    r"G:\PacBio\近观DNA_FRET\20261109\sample5_532nm_2mW_0p1S_2000frames_fov3.sif",
    r"G:\PacBio\近观DNA_FRET\20261109\sample6_532nm_2mW_0p1S_2000frames_fov1.sif",
    r"G:\PacBio\近观DNA_FRET\20261109\sample6_532nm_2mW_0p1S_2000frames_fov2.sif",
    r"G:\PacBio\近观DNA_FRET\20261109\sample6_532nm_2mW_0p1S_2000frames_fov3.sif",
]

# 只补算个别视野时, 把**文件名**填进来(留空 = 全部按 SIF_FILES 跑)。
# 参数改动后往往只需要重算一两个视野, 定位仍有缓存, 一条 5 分钟左右。
ONLY_FILES = []

# ---------------------------------------------------------------- 固定参数
SPLIT_X, ROWS = 256, (0, 512)
X_TRIM, Y_TRIM = 0, 50
K_SIGMA_SAMPLE = dict(green=2.0, red=1.4)
DET_KW = dict(edge=5, f_em=np.sqrt(2), bg_window=15, max_size=7, min_neighbors=2)
LOC_KW = dict(edge=5, F_EM=np.sqrt(2), use_pixel=False, use_gradient=True,
              n_jobs=14, verbose=False)
FILT_KW = dict(photons=(3, 1e7), bg=(0, 1e5), sigma=(0.7, 4.0), lp=None)
FRAMES_PER_SEG, PAD, EXTRA_TRIM = 10, 4, 0
RAD, EXCL, SMOOTH = 3, 3, 10
PSF_SIGMA_PX = 1.5      # 硬圆盘: σ 只写进字段, 不进核
G_WIN = 4

# γ 不再逐视野估计: 逐视野的 median(ΔI_A/ΔI_D) 在 0.5~1.5 之间跳, 只反映每个视野
# 里那几个大 ΔI_D 离群点, 不是仪器常数。固定成 1.1 之后所有视野的 E 才可比。
# 设成 None 就退回逐视野估计(每次照旧会打印估计值, 存进 summary.txt)。
GAMMA_FIXED = 1.1
# 想保住上一轮的结果就把这个改成 '_g1p1' 之类; 空串 = 覆盖 <名>_fret/
OUT_TAG = ''

EV = fr.EventSpec(sw=4, sw_long=20, lag_max=16, lag_step=2,
                  min_snr_on_a=3.0, off_ratio=0.25,
                  min_snr_a=4.0, min_snr_d=2.0,
                  min_snr_on_d=3.0, min_drop_d=150.0)
ES = fr.EfficiencySpec(pre_win=100, post_win=400, post_thr_d=100.0, min_on=10)
BG_GAP, BG_WIDTH = 2, 4
FILLERS = [tr.ApertureRaw(mode='hard', radius=RAD),
           tr.AnnulusRing(gap=BG_GAP, width=BG_WIDTH, excl_radius=EXCL,
                          bg_smooth=SMOOTH),
           tr.IntensityFromRawBg()]
APS_REPORT = (f'ApertureRaw(hard, r={RAD}) + AnnulusRing(+{BG_GAP}..'
              f'+{BG_GAP + BG_WIDTH} px, excl={EXCL}, smooth={SMOOTH})')


# ---------------------------------------------------------------- 日志
# redirect_stdout 之后 sys.stdout 就是 _KeepCacheLog, 在它里面 print 会递归,
# 所以一律写到事先留好的真实流 _REAL_OUT。
_REAL_OUT = sys.stdout
_T_FILE = 0.0       # 当前文件开始时刻


def _log(msg=''):
    print(msg, file=_REAL_OUT)


def _p(msg):
    """带耗时戳的进度行(相对当前文件开始)。"""
    _log(f'  {time.time() - _T_FILE:6.1f}s  {msg}')


class _KeepCacheLog(io.TextIOBase):
    """吃掉流程内部的 print, 只放行 `[cache] ...`(命中缓存 / 重新计算)。"""

    def __init__(self, drop=()):
        self._rest = ''
        self._drop = tuple(drop)

    def write(self, s):
        if not s:
            return 0
        self._rest += s
        *lines, self._rest = self._rest.split('\n')
        for L in lines:
            if L.startswith('[cache]'):
                t = L.replace('[cache] ', '')
                if not any(d in t for d in self._drop):
                    _log(f'  {time.time() - _T_FILE:6.1f}s  cache: {t}')
        return len(s)

    def flush(self):
        pass


def _save_fig(fig, path, dpi=300):
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)


# ---------------------------------------------------------------- 落盘: 成品
# 每个视野只留合作者要的东西; 上一轮留下的旧文件(旧图/标定/报表)先清掉,
# 否则重跑之后目录里会新旧混在一起。
# CSV 写入与亮度/寿命图都在 smlm.fret 里(io / viz), 这里只负责编排。
# 上一版/更早版本留下、现在不再产出的文件; 名字写全, 免得目录里新旧混在一起。
_STALE = ('traces_all.png', 'gamma.png', 'summary.txt', 'fretcal.npz',
          'brightness_green.csv', 'brightness_green.png',
          'brightness_red.csv', 'brightness_red.png', 'lifetime_red.png')


def _clean_stale(out_dir):
    for name in _STALE:
        p = os.path.join(out_dir, name)
        if os.path.exists(p):
            os.remove(p)



def _run_one(src, *, calib, save_tif=False):
    """Process one sif; returns a one-row summary dict."""
    global _T_FILE
    _T_FILE = time.time()

    src = os.path.abspath(str(src))
    base = os.path.splitext(src)[0]
    out_dir = base + '_fret' + OUT_TAG
    os.makedirs(out_dir, exist_ok=True)
    _clean_stale(out_dir)

    # ---- 1) 读 sif -> 光子换算 -> 裁上下 ----
    _p(f'读 sif ({os.path.getsize(src) / 2**30:.2f} GiB) ...')
    raw = load_raw(src)
    _p('光子换算 + 裁上下 ...')
    cam = EMCCD(conv_factor=5.75, em_gain=300)
    image = cam.process(raw, pixel_size=106.6)
    prep = Pipeline(Crop(y=ROWS, x=(0, 512)), SelectFrames(stop=None))
    stack = prep(image)
    del image
    _p(f'sif  : {raw.n_frames} 帧  {raw.exposure_time*1000:.0f} ms/帧  '
       f'-> {stack.data.shape[1:]}')
    if save_tif:
        from smlm.io import save_tif
        save_tif(raw.data, base + '_raw.tif')

    # ---- 2) 按标定取框 -> MultiImageStack ----
    CROPS = dict(zip(('g', 'r'),
                     mp.analysis_crops(calib, x_trim=X_TRIM, y_trim=Y_TRIM)))
    multi = mp.multi_from_crops(stack, CROPS, reference='g', reg={'r': calib},
                                margin=Y_TRIM, roi_dict=calib.roi,
                                verbose=False)
    _p(f'取框 : green y{CROPS["g"]["y"]} x{CROPS["g"]["x"]}   '
       f'red y{CROPS["r"]["y"]} x{CROPS["r"]["x"]}')

    # ---- 3) 双通道定位(带缓存) ----
    LOCS = {}
    for ch, ks, tag in (('g', K_SIGMA_SAMPLE['green'], 'green'),
                        ('r', K_SIGMA_SAMPLE['red'], 'red')):
        _p(f'定位 {tag} (k_sigma={ks}) ...')
        LOCS[ch] = mp.localize_channel(multi.channel(ch), k_sigma=ks,
                                       det_kw=DET_KW, loc_kw=LOC_KW,
                                       filt_kw=FILT_KW,
                                       src_path=f'{base}_{tag}', verbose=False,
                                       tag=tag)
        _p(f'定位 {tag}: {len(LOCS[ch])} locs')

    # ---- 4) 先配准点云 -> 估一条物理漂移 -> 按通道换算 -> 同尺寸裁切 ----
    _p('估漂移 + 校正 ...')
    drift = mp.estimate_drift_multi(multi, LOCS, frames_per_seg=FRAMES_PER_SEG,
                                   verbose=False)
    window = mp.drift_crop_window(drift, multi.shape[-2:], pad=PAD,
                                  extra_trim=EXTRA_TRIM)
    multi_c, locs_c = mp.correct_multi(multi, LOCS, drift, window=window,
                                       verbose=False)
    geom_c = multi_c.geometry(check_coverage=False)
    locs_g_c, locs_r_c = locs_c['g'], locs_c['r']
    corr_g, corr_r = multi_c.channel('g'), multi_c.channel('r')
    del stack, multi
    _p(f'漂移 : green {drift["g"].total_drift:.2f} px   '
       f'red {drift["r"].total_drift:.2f} px   -> 校正后 '
       f'{corr_g.data.shape[1:]}')

    N_FRAMES = int(corr_r.n_frames)
    LINK_KW = dict(search_radius=0.6, max_gap=1000, motion='static',
                   min_frames=3, first_frame_range=(0, N_FRAMES - 1),
                   ecc_p=1e-3, do_merge=True, r_merge=1.5)

    # ---- 5) emitter: 红通道连链 + 绿通道自己的分子(只用于背景挖洞) ----
    _p('连 emitter ...')
    emitters_r, _ = analyze_emitters_sm(locs_r_c, N_FRAMES, **LINK_KW)
    C_R = np.array([[e.centroid[1], e.centroid[0]] for e in emitters_r], float)
    C_G = geom_c.to_reference(C_R[:, ::-1], 'r')[:, ::-1]

    emitters_g_own, _ = analyze_emitters_sm(locs_g_c, N_FRAMES, **LINK_KW)
    MASK_G = np.array([[e.centroid[1], e.centroid[0]]
                       for e in emitters_g_own], float)
    _p(f'emitter: red {len(emitters_r)}   green(own) {len(emitters_g_own)}')

    # ---- 6) 取 trace ----
    _p(f'取 trace ({len(emitters_r)} 分子 x {N_FRAMES} 帧) ...')
    tr_r = tr.TraceFillPipeline(corr_r.data, C_R, sigma=PSF_SIGMA_PX,
                                n_frames=N_FRAMES, tag='red').run(FILLERS)
    # 绿通道的背景环用它**自己的** emitter 挖洞(只有供体、没有受体的分子)
    tr_g = tr.TraceFillPipeline(corr_g.data, C_G, centers_for_bg=MASK_G,
                                sigma=PSF_SIGMA_PX, n_frames=N_FRAMES,
                                tag='green').run(FILLERS)

    # ---- 7) 存全部(未筛选) trace ----
    # 与事件筛选无关, 所以放在算 E 之前 —— 即使一个事件都没通过也有成品。
    EX_T = float(raw.exposure_time)
    t = np.arange(N_FRAMES) * EX_T
    _p(f'存 traces_all.csv ({len(tr_g)} 分子 x {N_FRAMES} 帧) ...')
    fr.save_traces_csv(tr_g, tr_r, t, os.path.join(out_dir, 'traces_all.csv'),
                       verbose=False)

    # ---- 8) 扫台阶 + 标定 (α/β 照旧逐视野估, γ 用固定值) ----
    _p('扫台阶 + 标定 ...')
    STEP = fr.scan_steps(tr_r, tr_g, EV, verbose=False)
    GOOD = [e for e in STEP if fr.passed(e)]
    cal = fr.calibrate(STEP, traces_a=tr_r, traces_d=tr_g, win=G_WIN,
                       verbose=False)
    gamma_est = float(cal.gamma)          # 逐视野估计值, 只留作记录
    if GAMMA_FIXED is not None:
        cal.gamma = float(GAMMA_FIXED)
        cal.detail['gamma_estimated'] = gamma_est
    _p(f'事件 : {len(STEP)} 候选 -> {len(GOOD)} 合格')
    _p(f'标定 : gamma={cal.gamma:.3f}'
       f'{"(固定)" if GAMMA_FIXED is not None else "(估计)"}   '
       f'alpha={cal.alpha:.3f}  beta={cal.beta:.0f}   '
       f'[逐视野 γ 估计 {gamma_est:.3f}, n={cal.n_gamma}]')

    # ---- 9) 效率 ----
    _p('算 E ...')
    FRET, _ = fr.fret_efficiency(
        GOOD, tr_r, tr_g, cal, ES,
        t_bleach=lambda j: j * float(raw.exposure_time), verbose=False)

    out = dict(src=src, out_dir=out_dir, n_cand=len(STEP), n_good=len(GOOD),
               n_fret=len(FRET), gamma=cal.gamma, gamma_est=gamma_est,
               alpha=cal.alpha, beta=cal.beta, e_med=np.nan, e_p16=np.nan,
               e_p84=np.nan, n_e_gt1=0, n_neg=0, seconds=0.0)

    # ---- 10) 存 fret.csv + good_traces.csv (筛选过的 trace 全帧) ----
    # 没算出 E 时也写表头, 目录结构保持一致。
    _centres = {i: (float(C_G[i, 1]), float(C_G[i, 0]))
                for i in range(len(C_G))}
    fr.save_csv(FRET, os.path.join(out_dir, 'fret.csv'), cal, centres=_centres)
    _good = sorted({int(e.emitter_id) for e in GOOD})
    _p(f'存 good_traces.csv ({len(_good)} 分子 x {N_FRAMES} 帧) ...')
    fr.save_traces_csv([tr_g[i] for i in _good],
                       [tr_r[i] for i in _good], t,
                       os.path.join(out_dir, 'good_traces.csv'), verbose=False)
    if not FRET:
        _p('效率 : 没有能算出 E 的分子, 已写两个 csv 的表头')
        return out

    E = np.array([r.E for r in FRET], float)
    IDp = np.array([r.ID_pre for r in FRET], float)
    Eok = E[np.isfinite(E)]
    _p(f'效率 : n={len(FRET)}   E 中位 {np.median(Eok):.3f}   '
       f'E>1 {int((E > 1).sum())}   I_D^pre<0 {int((IDp < 0).sum())}')

    # ---- 11) 存图: 通过判据的 trace ----
    _p(f'画图 ({len(GOOD)} 条 trace) ...')
    _demo = sorted(GOOD, key=lambda e: -e.dIA / max(e.sig_a, 1e-9))
    NCOL = 6
    NROW = max(1, int(np.ceil(len(_demo) / NCOL)))
    fig, axs = plt.subplots(NROW, NCOL, figsize=(2.7 * NCOL, 1.85 * NROW),
                            squeeze=False)
    _al = float(cal.alpha) if np.isfinite(cal.alpha) else 0.0
    for k in range(NROW * NCOL):
        ax = axs[k // NCOL][k % NCOL]
        if k >= len(_demo):
            ax.axis('off')
            continue
        e = _demo[k]
        i, j = e.emitter_id, e.frame
        ax.axvspan(t[max(0, j - G_WIN)], t[j], color='0.86', lw=0)
        ax.axvspan(t[min(N_FRAMES - 1, j + e.lag)],
                   t[min(N_FRAMES - 1, j + e.lag + G_WIN)], color='0.72', lw=0)
        ax.plot(t, np.asarray(tr_g[i].intensity), color='green', lw=0.6,
                label='donor')
        ax.plot(t, np.asarray(tr_r[i].intensity), color='red', lw=0.6, alpha=0.85,
                label='acceptor')
        ax.axvline(t[j], color='k', ls='--', lw=0.8)
        ax.set_title(f'#{i}  j={j} lag={e.lag}\nΔI_A={e.dIA:.0f}  '
                     f'ΔI_D={e.dID:.0f}  γ={e.gamma_ratio + _al:.2f}',
                     fontsize=6.5, pad=2)
        ax.tick_params(labelsize=5, length=2)
        if k % NCOL == 0:
            ax.set_ylabel('photons', fontsize=6)
        if k // NCOL == NROW - 1:
            ax.set_xlabel('time (s)', fontsize=6)
    axs[0][0].legend(fontsize=6, loc='upper right', frameon=False)
    fig.suptitle(f'{os.path.basename(src)}   全部 {len(_demo)} 个合格事件   '
                 f'灰 = γ 读数窗 (±{G_WIN} 帧)', fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    _save_fig(fig, os.path.join(out_dir, 'good_traces_all.png'), dpi=300)

    # ---- 12) 存图: E 分布 ----
    ED = np.array([r.E_don for r in FRET], float)
    col = np.where(E > 1.0, 'crimson',
                   np.where(E < 0.0, 'royalblue', 'steelblue'))
    fig = plt.figure(figsize=(14, 4.2))
    gs = fig.add_gridspec(1, 3, wspace=0.28)

    ax = fig.add_subplot(gs[0, 0])
    lo = float(np.floor(min(E.min(), -0.05) * 10) / 10)
    hi = float(np.ceil(max(E.max(), 1.05) * 10) / 10)
    ax.hist(E, bins=np.linspace(lo, hi, 40), color='steelblue', alpha=0.85,
            edgecolor='0.4', lw=0.4)
    ax.axvline(1.0, color='r', ls='--', lw=1.2, label='E = 1')
    ax.axvline(np.median(E), color='k', ls=':', lw=1.2,
               label=f'median {np.median(E):.2f}')
    ax.set_xlabel('FRET efficiency E (raw)')
    ax.set_ylabel('molecules')
    ax.set_title(f'E 分布 (原始值, n={len(FRET)})', fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 1])
    ax.scatter(IDp, E, s=30, c=col, alpha=0.85, edgecolors='0.3', linewidths=0.3)
    ax.axhline(1.0, color='r', ls='--', lw=1.0)
    ax.axhline(0.0, color='k', ls=':', lw=0.9)
    ax.axvline(0.0, color='0.6', lw=0.9)
    ax.set_xlabel(r'$I_D^{pre}$ (photons)')
    ax.set_ylabel('E (raw)')
    ax.set_title(r'E 对 $I_D^{pre}$: 负的 pre 把 E 推到 >1(红) / <0(蓝)', fontsize=9)
    ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 2])
    ax.scatter(ED, E, s=30, c=col, alpha=0.85, edgecolors='0.3', linewidths=0.3)
    ax.plot([0, 1], [0, 1], 'k--', lw=0.9, label='一致')
    ax.axhline(1.0, color='r', ls='--', lw=1.0)
    ax.set_xlabel(r'$E_{don} = 1 - I_D^{pre}/I_D^{post}$')
    ax.set_ylabel('E (raw)')
    ax.set_title('两个独立估计的一致性', fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    fig.suptitle(f'{os.path.basename(src)}   smFRET raw  (γ={cal.gamma:.2f}, '
                 f'α={cal.alpha:.3f}, β={cal.beta:.0f};  n={len(FRET)})',
                 fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    _save_fig(fig, os.path.join(out_dir, 'E_distribution.png'), dpi=200)
    out.update(e_med=float(np.median(Eok)) if Eok.size else np.nan,
               e_p16=float(np.percentile(Eok, 16)) if Eok.size else np.nan,
               e_p84=float(np.percentile(Eok, 84)) if Eok.size else np.nan,
               n_e_gt1=int((E > 1).sum()), n_neg=int((IDp < 0).sum()))
    _p(f'保存 : {out_dir}')
    return out


def run_one(src, *, calib, save_tif=False):
    """跑一个 sif: 只放行 [cache] 提示, 其余流程内部 print 丢掉。"""
    with contextlib.redirect_stdout(_KeepCacheLog()):
        return _run_one(src, calib=calib, save_tif=save_tif)


# ---------------------------------------------------------------- 循环
calib_batch = mp.DualViewCalibration.load(os.path.join(
    os.path.dirname(os.path.abspath(SIF_FILES[0])), 'dualcal_v2.npz'))

# ONLY_FILES 里只写文件名, 按 basename 匹配
if ONLY_FILES:
    _want = {os.path.basename(x) for x in ONLY_FILES}
    todo = [f for f in SIF_FILES if os.path.basename(f) in _want]
    _miss = _want - {os.path.basename(f) for f in todo}
    if _miss:
        print('[!] ONLY_FILES 里这些文件名不在 SIF_FILES 中: %s' % sorted(_miss))
else:
    todo = list(SIF_FILES)

_log(f'γ    : ' + (f'固定 {GAMMA_FIXED} (不再逐视野估计)' if GAMMA_FIXED is not None
                   else '逐视野估计'))
_log(f'待处理 {len(todo)} / {len(SIF_FILES)} 个文件'
     + ('  (ONLY_FILES 已生效)' if ONLY_FILES else ''))

rows = []
t0_all = time.time()
for _k, _src in enumerate(todo, 1):
    _t = time.time()
    _log(f'\n[{_k}/{len(todo)}] {os.path.basename(_src)}')
    try:
        _r = run_one(_src, calib=calib_batch)
        _r['seconds'] = time.time() - _t
        rows.append(_r)
        _log(f'  ---- 完成: {_r["n_fret"]} 个分子   {_r["seconds"]:.0f}s')
        _log(f'       预计剩余 {_r["seconds"] * (len(todo) - _k) / 60:.0f} min')
    except Exception as _exc:
        rows.append(dict(src=_src, out_dir='', n_cand=0, n_good=0, n_fret=0,
                         gamma=GAMMA_FIXED, gamma_est=np.nan, alpha=np.nan,
                         beta=np.nan, e_med=np.nan, e_p16=np.nan, e_p84=np.nan,
                         n_e_gt1=0, n_neg=0, seconds=time.time() - _t,
                         error=repr(_exc)))
        _log(f'  ---- FAILED: {_exc!r}')
    finally:
        gc.collect()

# ---------------------------------------------------------------- 汇总
_log()
_log(f'{"file":<44}{"mol":>6}{"gamma":>8}{"a-est":>8}{"alpha":>8}{"E med":>8}'
     f'{"E>1":>5}{"time":>8}')
_log('-' * 95)
for _r in rows:
    _name = os.path.basename(_r['src'])[:43]
    if _r.get('error'):
        _log(f'{_name:<44}{"--":>6}{"FAILED":>8}')
        continue
    _log(f'{_name:<44}{_r["n_fret"]:>6}{_r["gamma"]:>8.3f}'
         f'{_r["gamma_est"]:>8.3f}{_r["alpha"]:>8.3f}{_r["e_med"]:>8.3f}'
         f'{_r["n_e_gt1"]:>5}{_r["seconds"]:>7.0f}s')

_ok = [r for r in rows if not r.get('error') and r['n_fret']]
if _ok:
    _meds = np.array([r['e_med'] for r in _ok], float)
    _log(f'\n共 {len(rows)} 个文件 (成功 {len(_ok)}), 总耗时 {time.time() - t0_all:.0f}s')
    _log(f'合计 {sum(r["n_fret"] for r in _ok)} 个分子   '
         f'各视野 E 中位: 中位 {np.median(_meds):.3f}   '
         f'范围 {_meds.min():.3f}~{_meds.max():.3f}')
else:
    _log(f'\n共 {len(rows)} 个文件, 全部失败, 总耗时 {time.time() - t0_all:.0f}s')


## 分组统计已算好的 FRET 效率

只读各视野 `<sif>_fret/fret.csv`，**不重跑**定位/取 trace。

分组：`sample1 + sample4`（第一组）、`sample2 + sample5`（第二组）、`sample6`（第三组）。
每组把所有 FOV 的分子并在一起，两端各裁掉异常值后画一个分布图。

裁剪方式由 `TRIM` 控制：

| 取值 | 规则 | 说明 |
|---|---|---|
| `'tukey'`（默认） | `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]` | 稳健、数据驱动，两端同时裁 |
| `'pct'` | 两侧各裁 `TRIM_PCT` % | 想按固定百分位裁就用这个 |
| `None` | 不裁 | 只看原始分布 |

> `E > 1` 的分子来自 `I_D^pre ≈ 0`（供体淬灭到探测限以下），不是污染，所以裁剪后仍会留下一部分，表里会单独报告它的比例。

In [ ]:
# ============ 分组统计已算好的 FRET 效率 (只读 fret.csv, 不重跑) ============
# 输出统一归档到 <ROOT>/FRET_summary/:
#   README.md             方法与参数的完整说明(从 trace 筛选开始)
#   E_groups.png          三组 E 分布
#   E_per_molecule.csv    逐分子明细(裁剪后, 含组别/来源/γ/S_A)
#   E_group_stats.csv     每组汇总统计量
import os
import re
import csv
import glob
import datetime

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

ROOT = r"G:\PacBio\近观DNA_FRET\20261109"
OUT = os.path.join(ROOT, 'FRET_summary')
os.makedirs(OUT, exist_ok=True)

PNG_OUT = os.path.join(OUT, 'E_groups.png')
CSV_MOL = os.path.join(OUT, 'E_per_molecule.csv')
CSV_STATS = os.path.join(OUT, 'E_group_stats.csv')
MD_OUT = os.path.join(OUT, 'README.md')

# 分组: (图例名, 属于该组的样品前缀)
GROUPS = [("sample1+4", ["sample1", "sample4"]),
          ("sample2+5", ["sample2", "sample5"]),
          ("sample6",   ["sample6"])]

# 异常值裁剪: 'tukey' = [Q1-1.5IQR, Q3+1.5IQR]; 'pct' = 两侧各裁 TRIM_PCT%;
#              None    = 不裁
TRIM = 'tukey'
TRIM_PCT = 1.0


# ---------------------------------------------------------------- 工具
def _read_rows(path):
    with open(path, newline='', encoding='utf-8-sig') as fh:
        return list(csv.DictReader(fh))


def _tofloat(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan


def _col(rows, key):
    return np.asarray([_tofloat(r.get(key, '')) for r in rows], float)


def _bounds(x):
    """两端都裁。x 必须已是去掉 nan 的一维数组。"""
    if TRIM is None:
        return float(x.min()), float(x.max())
    if str(TRIM).lower() == 'tukey':
        q1, q3 = np.percentile(x, [25, 75])
        iqr = q3 - q1
        return float(q1 - 1.5 * iqr), float(q3 + 1.5 * iqr)
    lo, hi = np.percentile(x, [TRIM_PCT, 100.0 - TRIM_PCT])
    return float(lo), float(hi)


def _summ_values(files, key, fallback=None):
    """汇总某个参数字段的取值(去重)。

    ``summary.txt`` 已不再落盘, 所以取不到文件时退回 ``fallback`` —— 通常是本内核
    里与批量单元同源的那个参数对象; 老结果目录里还留着 summary.txt 时以文件为准。
    """
    vals = set()
    for f in files:
        p = os.path.join(os.path.dirname(f), 'summary.txt')
        if not os.path.exists(p):
            continue
        with open(p, encoding='utf-8') as fh:
            for L in fh:
                if ':' in L:
                    k, v = L.split(':', 1)
                    if ' '.join(k.split()) == key:
                        vals.add(v.strip())
    if not vals and fallback:
        vals.add(str(fallback))
    return sorted(vals)


# ---------------------------------------------------------------- 归组
_files = sorted(glob.glob(os.path.join(ROOT, '*_fret', 'fret.csv')))
if not _files:
    raise RuntimeError(ROOT + ' 下没有找到任何 *_fret/fret.csv')

_per_sample = {}
for _fp in _files:
    _name = os.path.basename(os.path.dirname(_fp))
    _m = re.match(r'(sample\d+)', _name)
    if _m:
        _per_sample.setdefault(_m.group(1), []).append((_name, _fp))

print(f'找到 {len(_files)} 个 fret.csv, 覆盖样品: {sorted(_per_sample)}\n')

pooled, stats, used_files = [], [], []
for _label, _members in GROUPS:
    _chunks, _rows_kept, _gam, _grp_files = [], [], set(), []
    for _s in _members:
        for _name, _fp in _per_sample.get(_s, []):
            _grp_files.append(_fp)
            _chunks.append(_col(_read_rows(_fp), 'E'))
            _gam.update(float(g) for g in np.unique(_col(_read_rows(_fp), 'gamma'))
                        if np.isfinite(g))

    if not _chunks:
        print(f'[!] {_label}: 没有数据, 跳过')
        continue

    _E = np.concatenate(_chunks)
    _E = _E[np.isfinite(_E)]
    _lo, _hi = _bounds(_E)
    _Ek = _E[(_E >= _lo) & (_E <= _hi)]
    if _Ek.size < 3:
        print(f'[!] {_label}: 裁剪后只剩 {_Ek.size} 个点, 跳过')
        continue

    # 明细: 只保留落在裁剪区间内的分子
    _n_fov = set()
    for _s in _members:
        for _name, _fp in _per_sample.get(_s, []):
            for _r in _read_rows(_fp):
                _v = _tofloat(_r.get('E', ''))
                if np.isfinite(_v) and _lo <= _v <= _hi:
                    _rows_kept.append(dict(_r, _group=_label, _src=_name))
                    _n_fov.add(_name)

    used_files += _grp_files
    pooled.append(_Ek)
    stats.append(dict(label=_label, n_raw=int(_E.size), n=int(_Ek.size),
                      lo=_lo, hi=_hi, n_fov=len(_n_fov),
                      med=float(np.median(_Ek)),
                      p16=float(np.percentile(_Ek, 16)),
                      p84=float(np.percentile(_Ek, 84)),
                      mean=float(_Ek.mean()),
                      std=float(_Ek.std(ddof=1)),
                      gt1=int((_Ek > 1).sum()),
                      gammas=sorted(_gam), rows=_rows_kept,
                      files=_grp_files))

# ---------------------------------------------------------------- 屏幕报表
print('异常值裁剪: %s%s' % (TRIM, ('  两侧各 %g%%' % TRIM_PCT)
                          if str(TRIM).lower() == 'pct' else ''))
hdr = ('%-12s%6s%8s%8s%22s%9s%18s%20s%8s'
       % ('组', '视野', 'n原始', 'n裁剪', '裁剪区间', '中位',
          'p16~p84', '均值±std', 'E>1'))
print(hdr)
print('-' * 108)
for st in stats:
    print('%-12s%6d%8d%8d%22s%9.3f%18s%20s%7.0f%%'
          % (st['label'], st['n_fov'], st['n_raw'], st['n'],
             '[%.3f, %.3f]' % (st['lo'], st['hi']), st['med'],
             '%.3f~%.3f' % (st['p16'], st['p84']),
             '%.3f±%.3f' % (st['mean'], st['std']),
             st['gt1'] / st['n'] * 100.0))

for st in stats:
    if len(st['gammas']) > 1:
        print(f'[!] {st["label"]} 组内 γ 不统一: {st["gammas"]}')

# ---------------------------------------------------------------- 画图
lo = min(s['lo'] for s in stats)
hi = max(s['hi'] for s in stats)
BINS = np.linspace(lo, hi, 41)

fig, axs = plt.subplots(1, len(stats), figsize=(5.2 * len(stats), 4.2),
                        sharex=True, sharey=True, squeeze=False)
for ax, st, Ek in zip(axs[0], stats, pooled):
    ax.hist(Ek, bins=BINS, color='steelblue', alpha=0.85, edgecolor='0.35', lw=0.4)
    ax.axvspan(st['p16'], st['p84'], color='orange', alpha=0.15, lw=0)
    ax.axvline(st['med'], color='k', ls='--', lw=1.3,
               label=f"median {st['med']:.3f}")
    ax.axvline(1.0, color='r', ls=':', lw=1.0, label='E = 1')
    ax.set_title(f"{st['label']}   n = {st['n']}   "
                 f"p16~p84 {st['p16']:.2f}~{st['p84']:.2f}", fontsize=10)
    ax.set_xlabel('FRET efficiency E')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.25)
axs[0][0].set_ylabel('molecules')
fig.suptitle(f'FRET efficiency per group  (各视野合并, 两端裁剪: {TRIM})',
             fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(PNG_OUT, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

# ---------------------------------------------------------------- 存明细
_rows = [r for st in stats for r in st['rows']]
_keys = ['_group', '_src', 'idx', 'E', 'E_don', 't_bleach_s', 'gamma',
         'I_A_pre', 'I_D_pre', 'I_D_post', 'S_A']
with open(CSV_MOL, 'w', newline='', encoding='utf-8-sig') as fh:
    w = csv.DictWriter(fh, fieldnames=_keys, extrasaction='ignore')
    w.writeheader()
    for r in sorted(_rows, key=lambda r: (r['_group'], r['_src'],
                                          _tofloat(r['idx']))):
        w.writerow(r)

with open(CSV_STATS, 'w', newline='', encoding='utf-8-sig') as fh:
    w = csv.writer(fh)
    w.writerow(['group', 'n_fov', 'n_raw', 'n_kept', 'trim_lo', 'trim_hi',
                'median', 'p16', 'p84', 'mean', 'std', 'frac_E_gt1',
                'gamma_list'])
    for st in stats:
        w.writerow([st['label'], st['n_fov'], st['n_raw'], st['n'],
                    round(st['lo'], 4), round(st['hi'], 4),
                    round(st['med'], 4), round(st['p16'], 4),
                    round(st['p84'], 4), round(st['mean'], 4),
                    round(st['std'], 4), round(st['gt1'] / st['n'], 4),
                    ';'.join('%.4f' % g for g in st['gammas'])])

# ---------------------------------------------------------------- 写 README
# summary.txt 不再落盘: 批量单元与本单元同一内核时, 直接用它的参数对象填表。
_fb = {'aperture': globals().get('APS_REPORT')}
for _k, _obj in (('event spec', globals().get('EV')),
                 ('efficiency', globals().get('ES'))):
    _report = getattr(_obj, 'report', None)
    if _report is not None:
        _fb[_k] = _report()
if 'G_WIN' in globals():
    _fb['gamma win'] = f'+/-{G_WIN} frames around the bleach step'
if globals().get('GAMMA_FIXED') is not None:
    _fb['gamma src'] = f'FIXED {float(GAMMA_FIXED):.4f}'

_ap = _summ_values(used_files, 'aperture', _fb.get('aperture'))
_ev = _summ_values(used_files, 'event spec', _fb.get('event spec'))
_es = _summ_values(used_files, 'efficiency', _fb.get('efficiency'))
_gw = _summ_values(used_files, 'gamma win', _fb.get('gamma win'))
_gs = _summ_values(used_files, 'gamma src', _fb.get('gamma src'))
_gs_fixed = sorted({x.split('estimate')[0].strip() for x in _gs})
_gs_txt = ('; '.join(_gs_fixed) + f'  x{len(_gs)}') if _gs else '(未读到)'
_all_a = _col(_rows, 'alpha')
_all_b = _col(_rows, 'beta')


def _one_or_list(vals, empty='(见 §2)'):
    return '; '.join(vals) if vals else empty


tbl_res = ['| 组 | 视野数 | n(原始) | n(裁剪) | 裁剪区间 | 中位 | p16~p84 | 均值±std | E>1 占比 |',
           '|---|---|---|---|---|---|---|---|---|']
for st in stats:
    tbl_res.append(
        f"| {st['label']} | {st['n_fov']} | {st['n_raw']} | {st['n']} | "
        f"[{st['lo']:.3f}, {st['hi']:.3f}] | **{st['med']:.3f}** | "
        f"{st['p16']:.3f}~{st['p84']:.3f} | {st['mean']:.3f}±{st['std']:.3f} | "
        f"{st['gt1'] / st['n'] * 100:.0f}% |")

tbl_fov = ['| 来源 fret.csv | 组 |', '|---|---|']
for st in stats:
    for f in st['files']:
        dn = os.path.basename(os.path.dirname(f))
        tbl_fov.append(f"| `{dn}/fret.csv` | {st['label']} |")

n_fov_txt = '、'.join(str(s['n_fov']) for s in stats)

_md = [
    '# FRET 效率整合数据说明',
    '',
    f"生成时间: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}",
    f"数据目录: `{ROOT}`",
    '本文档由 notebook `test_batch_dual.ipynb` 的「分组统计」单元自动生成。',
    '',
    '---',
    '',
    '## 1. 这个目录里有什么',
    '',
    '| 文件 | 内容 |',
    '| --- | --- |',
    '| `E_groups.png` | 三组 FRET 效率分布（直方图，共享横轴；橙带为 p16~p84） |',
    '| `E_per_molecule.csv` | **逐分子明细**（已裁剪）：组别、来源视野、E、E_don、漂白时刻、γ/α/β、四个原始强度 |',
    '| `E_group_stats.csv` | 每组统计量：n、裁剪区间、中位、p16/p84、均值/std、E>1 比例 |',
    '| `README.md` | 本文件：方法与参数 |',
    '',
    '---',
    '',
    '## 2. 从 trace 筛选开始的完整流程',
    '',
    '### 2.1 图像与坐标',
    '',
    '- 数据：双视角 sif，532 nm 激发、2 mW、每帧 100 ms、2000 帧（**总时长 200 s**），512×512。',
    '- 分界列 256：左半为绿（供体）通道，右半为红（受体）通道。',
    '- 分析窗口：绿 `y(50,462) x(14,242)`，红 = 绿 + 256。上下各留 50 px 给漂移。',
    '- 漂移：先把两通道点云**配准到同一坐标系再合并**，估出**一条**物理漂移（绝不能各自估一条），',
    '  再用亚像素傅里叶平移同时校正图像与定位；校正后两通道同尺寸、同局部坐标原点。',
    '- 本批 13 个视野的累计漂移约 5.7 px。',
    '',
    '### 2.2 定位（trace 的输入）',
    '',
    '- `LocalMaxDetector`：k_sigma 绿 2.0 / 红 1.4（样品很暗，阈值必须放宽），edge=5、bg_window=15、max_size=7。',
    '- `MLELocalizer`：高斯拟合，use_gradient=True。',
    '- 过滤：photons 3~1e7、bg 0~1e5、sigma 0.7~4.0。',
    '- 结果缓存成 `<sif>_green.locs.npz` / `_red.locs.npz`（同参数重跑直接读盘）。',
    '',
    '### 2.3 连 emitter（一条 trace = 一个受体分子）',
    '',
    '- **只用红通道定位**连链：emitter 的定义就是「受体分子的闪烁轨迹」。',
    '- 参数：search_radius=0.6 px、max_gap=1000 帧（允许长暗间隙）、motion=static、',
    '  min_frames=3、ecc_p=1e-3（剔除拉长的假轨迹）、do_merge=True + r_merge=1.5。',
    '- 每个视野约 370 个 emitter（例：fov6 由 7361 条原始链 → 合并 2017 → 444 → 370）。',
    '',
    '### 2.4 取双通道 trace（**筛选的核心**）',
    '',
    '- **孔径**：硬圆盘 **r = 3 px**（7×7 见方内的圆盘，29 个像素），两个通道相同。',
    '  - 为什么不用匹配滤波：实测 MLE 的 σ 与亮度强相关（绿：暗 1.42 → 亮 2.78；红暗端 0.73',
    '    直接压在滤镜下限 0.7 上）。那个 σ 是「SNR 不足 + 两个分子叠在一起」的产物，不是 PSF 宽度；',
    '    σ 核与真实 σ 不符时匹配滤波**有偏**（偏差 = 2σ_核²/(σ_核²+σ_真²)）。硬圆盘不依赖 σ。',
    '  - 代价：收集率 f = 1−exp(−r²/2σ²) 与 σ 有关（r=3 时红 0.89 / 绿 0.79），',
    '    但两通道之**比**是常数，被 γ 一起吸收，中位 E 不受影响。',
    '- **中心**：红通道 emitter 质心，经配准映射到绿通道 → 两通道看同一个物理位置。',
    '- **背景**：环形估计，取 r+2 ~ r+6 px（6~9 px）的环；每个邻居挖掉 r ≤ 3 px 的核心；',
    '  环内做 3σ 裁剪后取**中位数**，得到**逐帧**背景序列，再套 10 帧居中滑动平均。',
    '  - 为什么逐帧：实测背景首/尾比（early/late）绿 1.11 / 红 1.06，是真实的背景衰减，',
    '    逐帧扣才跟得住；换成「只取末端常数」会把真实漂移抹掉。',
    '  - 曾尝试「减掉邻居 PSF 翅膀」：实测只占环的 1.15%（绿）/ 0.43%（红），',
    '    无法解释那个 11% 的趋势，已弃用。',
    '- trace 强度 = (Σ孔径内像素 − bg×孔径面积) / 归一化；噪声用 MAD 估计。',
    '',
    '### 2.5 事件扫描（trace → 候选事件）',
    '',
    '- 在一条双色 trace 上找「红降 + 绿升」同时最显著的那一帧 j；供体允许滞后 0~16 帧（步长 2）。',
    '- 短窗 4 帧、长窗 20 帧（用于判断抬升是否持续）。',
    '- 7 条判据全部通过才留下：',
    '  1. 台阶前受体在发光（> 3σ_A）',
    '  2. 台阶后受体灭掉（< 25% 或 < 3σ_A）',
    '  3. 红降显著（ΔI_A > 4σ_A）',
    '  4. 绿升显著（ΔI_D > 2σ_D）',
    '  5. 绿升在长窗内仍持续（> 2σ_D）',
    '  6. 去淬灭后供体在发光（> 3σ_D）',
    '  7. 绿升绝对幅度 ΔI_D > 150 photon（防止 γ 的分母是噪声）',
    '- 每个视野 370 个候选 → 约 47 个全过。',
    '',
    '### 2.6 标定 α / β / γ',
    '',
    '- **α（串色）** = median(I_A^post / I_D^post)：受体已灭、供体仍亮那一截的红/绿比，逐视野估计。',
    '- **β（受体通道底）** = median(I_A^post − α·I_D^post)，逐视野估计。',
    '- **γ（通道增益比）固定为 1.1**，不再逐视野估计：逐视野的 median(ΔI_A/ΔI_D) 在 0.72~1.42',
    '  之间跳，只反映每个视野里那几个 ΔI_D 很小的离群点，不是仪器常数；固定后各视野的 E 才可比。',
    '  逐视野的估计值只留在批量单元的控制台汇总表里（`a-est` 列），仅供追溯。',
    '',
    '### 2.7 计算 FRET 效率 E',
    '',
    '```',
    'S_A = I_A^pre − α·I_D^pre − β',
    'E   = S_A / (S_A + γ·I_D^pre)',
    '```',
    '',
    '- `pre` 窗口：台阶左侧最多 100 帧（不足就从第 0 帧开始）。',
    '- `post` 窗口：台阶右侧最多 400 帧，且**只平均绿光 > 100 photon 的帧**（至少 10 帧）——',
    '  受体漂白后供体去淬灭会再闪烁，暗帧会把 I_D^post 压低。',
    '- **不做任何截断**：I_D^pre 为负就让它为负，E 算出 >1 或 <0 都原样保留。',
    '- 同时给出独立的供体侧估计 `E_don = 1 − I_D^pre/I_D^post`，用于交叉检查。',
    '',
    '### 2.8 分组与异常值裁剪',
    '',
    '- 分组：`sample1 + sample4`、`sample2 + sample5`、`sample6`；每组把所有视野的分子并在一起。',
    f"- 裁剪方式：`{TRIM}`，两端同时裁。原始数据里存在 E = −55、+10 这类荒唐值",
    '  （`I_D^pre` 极小的分子，分母接近 0）。',
    f'- 组内按**分子数**加权（不是按视野等权），三组分别有 {n_fov_txt} 个视野。',
    '',
    '---',
    '',
    '## 3. 结果',
    '',
] + tbl_res + [
    '',
    '![E distribution](E_groups.png)',
    '',
    '---',
    '',
    '## 4. 实际使用的参数（来自各视野 fret.csv / 批量处理单元）',
    '',
    '| 项目 | 取值 |',
    '| --- | --- |',
    f'| 孔径 | {_one_or_list(_ap)} |',
    f'| 事件判据 | {_one_or_list(_ev)} |',
    f'| γ 读数窗 | {_one_or_list(_gw)} |',
    f'| 效率窗口 | {_one_or_list(_es)} |',
    f'| γ 来源 | {_gs_txt} |',
    '',
    f'本批 α 中位 {np.nanmedian(_all_a):.3f}、β 中位 {np.nanmedian(_all_b):.1f} photon'
    '（逐视野估计，逐分子写在 fret.csv 的 alpha / beta 列）。',
    '',
    '---',
    '',
    '## 5. 参与统计的视野',
    '',
] + tbl_fov + [
    '',
    '---',
    '',
    '## 6. 已知局限（读图前请先看这几条）',
    '',
    '1. **E > 1 不是污染。** 它来自 `I_D^pre ≈ 0`：供体在受体漂白前被完全淬灭，',
    '   真实信号已经到探测限以下（`I_D^pre` 只有几光子，噪声却有 ±40），',
    '   所以 `E = S_A/(S_A+γ·I_D^pre)` 的分母在 0 附近，算出 1.0~1.5 属正常。',
    '   换句话说这些分子的 E 只能给出**下界**，不是可以继续平均的数值。',
    '2. **组内按分子数加权。** sample1+4 中 sample4 占多数，sample2+5 中 sample2 占 86%，',
    '   因此这两组实质上是「以 sample4 为主 / 以 sample2 为主」。若想按视野等权，',
    '   应先算每个视野的中位数再加权。',
    '3. **窗口截断。** 每个视野只有 200 s，寿命长于 200 s 的分子看不到；',
    '   观测到的是「窗口内的残余寿命」。',
    '4. **选择偏倚。** 7 条判据要求「绿升 > 150 photon」且「台阶前受体在发光」，',
    '   天然偏向亮的、E 高的分子；低 E 与很暗的分子会被系统性排除。',
    '5. **σ 不等于 PSF 宽度。** 硬圆盘孔径不依赖 σ，但 E 的绝对标度依赖 γ = 1.1 这个假设值。',
    '',
    '---',
    '',
    '## 7. 如何复现',
    '',
    '1. 打开 `test_batch_dual.ipynb`；',
    '2. 跑「批量处理」单元（`SIF_FILES` 里是 13 个视野；只想补算个别视野就把文件名填进',
    '   `ONLY_FILES`），每个视野会生成一个 `<sif 名>_fret/` 目录',
    '   （fret.csv、E_distribution.png、good_traces_all.png、traces_all.csv、',
    '   good_traces.csv）；',
    '3. 跑「分组统计」单元，即本目录的内容。',
    '',
    '关键参数集中在「批量处理」单元顶部：`RAD`（孔径半径）、`EXCL`（邻居挖洞半径）、',
    '`SMOOTH`（背景滑动平均窗）、`GAMMA_FIXED`（固定 γ）、`EV`（事件判据）、`ES`（效率窗口）。',
    '',
]
with open(MD_OUT, 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(_md) + '\n')

# ---------------------------------------------------------------- 旧的散落文件清掉
for _old in (os.path.join(ROOT, 'E_groups.png'), os.path.join(ROOT, 'E_groups.csv')):
    if os.path.exists(_old):
        os.remove(_old)

print(f'\n归档目录: {OUT}')
for _n in sorted(os.listdir(OUT)):
    print('  %-24s %8.1f kB'
          % (_n, os.path.getsize(os.path.join(OUT, _n)) / 1024))
